# 03 - HDF5 and one `load_cube()` for both formats


## 1. Why HDF5 as well as GeoTIFF?

| | GeoTIFF | HDF5 |
|---|---|---|
| Strength | Standard for geospatial data, any GIS tool opens it | Fast random access to **pieces** of a big array |
| Layout on disk | `(bands, H, W)` | whatever we choose, we use `(H, W, B)` |
| Extra data | tags | any number of **datasets** and **attributes** in one file |

An HDF5 file works like a small file system: **datasets** are arrays (`cube`, `labels`, `wavelengths_nm`) and **attributes** are small notes attached to the file or a dataset (CRS, transform).

**Chunking:** instead of one huge block, the array is stored in small tiles (we use 16 x 16 pixels x all bands). Reading a 16 x 16 patch then touches only a few tiles. Our 3D-CNN will train on small patches, so this is the access pattern we want.

**Compression** (`gzip`) shrinks each chunk on disk. Reading decompresses it automatically.

A **convention** to remember: in memory the cube is always `(H, W, B)`. GeoTIFF stores `(B, H, W)`, so `load_cube()` transposes for us.

## 2. Imports and paths

In [9]:
from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import rasterio
from affine import Affine
from rasterio.crs import CRS
from rasterio.windows import Window

REPO_ROOT = Path(".") if Path("notebooks").is_dir() else Path("..")
MOCK_DIR = REPO_ROOT / "data" / "mock"

tif_path = MOCK_DIR / "mock_cube.tif"
labels_tif_path = MOCK_DIR / "mock_labels.tif"
h5_path = MOCK_DIR / "mock_cube.h5"

assert tif_path.exists(), "run notebook 02 first to create the mock GeoTIFFs"

## 3. Read the GeoTIFF the long way (once)

Before wrapping this in a function we do it by hand, so each step is visible. Remember the transpose from `(B, H, W)` to `(H, W, B)`, and that the wavelengths live in a tag.

In [10]:
with rasterio.open(tif_path) as src:
    cube = np.transpose(src.read(), (1, 2, 0))                  # (B, H, W) -> (H, W, B)
    wavelengths = np.array(src.tags()["wavelengths_nm"].split(","), dtype=np.float32)
    crs, transform = src.crs, src.transform

with rasterio.open(labels_tif_path) as src:
    labels = src.read(1)

H, W, B = cube.shape
print("cube:", cube.shape, cube.dtype, "| labels:", labels.shape, labels.dtype, "| wavelengths:", wavelengths.shape)
print("CRS:", crs)

cube: (67, 67, 200) float32 | labels: (67, 67) uint8 | wavelengths: (200,)
CRS: EPSG:32643


## 4. Write the HDF5 file

- `chunks=(16, 16, B)` means each tile holds 16 x 16 pixels with **all** bands, because a model needs the whole spectrum of every pixel in a patch.
- The CRS is stored as **WKT text** (a full, unambiguous description) and the transform as its **six numbers**, so the georeferencing travels with the data.

In [11]:
with h5py.File(h5_path, "w") as f:
    f.create_dataset("cube", data=cube, chunks=(16, 16, B), compression="gzip", compression_opts=4)
    f.create_dataset("labels", data=labels, compression="gzip")
    f.create_dataset("wavelengths_nm", data=wavelengths)

    f.attrs["layout"] = "HWB"
    f.attrs["crs_wkt"] = crs.to_wkt()
    f.attrs["transform"] = np.array(tuple(transform)[:6], dtype=np.float64)    # a, b, c, d, e, f

print(f"{tif_path.name}: {tif_path.stat().st_size / 1e6:.2f} MB")
print(f"{h5_path.name} : {h5_path.stat().st_size / 1e6:.2f} MB")

mock_cube.tif: 3.19 MB
mock_cube.h5 : 3.18 MB


## 5. Look inside the file

`visititems` walks the file like a folder listing. Each dataset reports its shape, dtype, chunk shape and compression.

In [12]:
def describe(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"{name:<16} shape={obj.shape}  dtype={obj.dtype}  chunks={obj.chunks}  compression={obj.compression}")


with h5py.File(h5_path, "r") as f:
    f.visititems(describe)
    print()
    for key, value in f.attrs.items():
        print(f"attr {key:<10}", str(value)[:70])

cube             shape=(67, 67, 200)  dtype=float32  chunks=(16, 16, 200)  compression=gzip
labels           shape=(67, 67)  dtype=uint8  chunks=(67, 67)  compression=gzip
wavelengths_nm   shape=(200,)  dtype=float32  chunks=None  compression=None

attr crs_wkt    PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHERO
attr layout     HWB
attr transform  [ 3.00e+01  0.00e+00  5.50e+05  0.00e+00 -3.00e+01  3.46e+06]


## 6. Read just a piece

Slicing an HDF5 dataset reads **only the chunks it needs**, not the whole array. Here we cut the same 16 x 16 window from both files and check the values match exactly.

In [13]:
with h5py.File(h5_path, "r") as f:
    patch_h5 = f["cube"][30:46, 10:26, :]                        # rows 30-45, cols 10-25, all bands

with rasterio.open(tif_path) as src:
    patch_tif = np.transpose(src.read(window=Window(col_off=10, row_off=30, width=16, height=16)), (1, 2, 0))

print("HDF5 patch:", patch_h5.shape, "| GeoTIFF patch:", patch_tif.shape)
print("identical :", np.array_equal(patch_h5, patch_tif))

HDF5 patch: (16, 16, 200) | GeoTIFF patch: (16, 16, 200)
identical : True


## 7. One `load_cube()` for both formats

The function looks at the file extension and returns the **same kind of object** either way, so the rest of the project never needs to know which format a cube came from. The `Cube` container keeps the pixel data together with its wavelengths and georeferencing.

Note `np.ascontiguousarray`: the GeoTIFF branch ends with a transposed *view*, and we want a normal C-ordered array that PyTorch and NumPy handle efficiently.

In [14]:
@dataclass
class Cube:
    data: np.ndarray                   # (H, W, B) float32
    wavelengths: np.ndarray            # (B,) nm
    crs: CRS
    transform: Affine
    labels: np.ndarray | None = None   # (H, W) uint8, or None if not available


def load_cube(path, labels_path=None):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in (".tif", ".tiff"):
        with rasterio.open(path) as src:
            data = np.transpose(src.read(), (1, 2, 0))           # (B, H, W) -> (H, W, B)
            wavelengths = np.array(src.tags()["wavelengths_nm"].split(","), dtype=np.float32)
            crs, transform = src.crs, src.transform
        labels = None
        if labels_path is not None:
            with rasterio.open(labels_path) as lab:
                labels = lab.read(1)

    elif suffix in (".h5", ".hdf5"):
        with h5py.File(path, "r") as f:
            data = f["cube"][...]
            wavelengths = f["wavelengths_nm"][...]
            crs = CRS.from_wkt(f.attrs["crs_wkt"])
            transform = Affine(*f.attrs["transform"])
            labels = f["labels"][...] if "labels" in f else None

    else:
        raise ValueError(f"Unsupported file type: {suffix!r} (use .tif or .h5)")

    return Cube(np.ascontiguousarray(data), wavelengths, crs, transform, labels)

## 8. Test it

The check that matters: both formats must give **identical** pixel values, wavelengths, labels, CRS and transform.

In [15]:
a = load_cube(tif_path, labels_path=labels_tif_path)
b = load_cube(h5_path)

print("GeoTIFF:", a.data.shape, a.data.dtype, "| HDF5:", b.data.shape, b.data.dtype)
print()
print("pixel values identical :", np.array_equal(a.data, b.data))
print("wavelengths identical  :", np.array_equal(a.wavelengths, b.wavelengths))
print("labels identical       :", np.array_equal(a.labels, b.labels))
print("CRS identical          :", a.crs == b.crs)
print("transform identical    :", a.transform == b.transform)
print("C-contiguous           :", a.data.flags["C_CONTIGUOUS"], b.data.flags["C_CONTIGUOUS"])

# an unknown file type should fail loudly with a clear message
try:
    load_cube("cube.npy")
except ValueError as err:
    print()
    print("error as expected:", err)

GeoTIFF: (67, 67, 200) float32 | HDF5: (67, 67, 200) float32

pixel values identical : True
wavelengths identical  : True
labels identical       : True
CRS identical          : True
transform identical    : True
C-contiguous           : True True

error as expected: Unsupported file type: '.npy' (use .tif or .h5)


## Takeaways

- **HDF5** = datasets + attributes in one file, with **chunking** (fast patch reads) and **compression**.
- Chunk shape should match how you read: small spatial tiles with **all bands** for patch-based models.
- Store georeferencing next to the data: **CRS as WKT**, **transform as six numbers**.
- Conventions: in memory `(H, W, B)`; GeoTIFF on disk `(B, H, W)`; HDF5 on disk `(H, W, B)`.
- One `load_cube()` hides the format difference, and a test proves both paths agree.

**Later:** when the FastAPI service needs `load_cube()`, move it into a small importable module, because an API cannot import from a notebook. Until then it lives here.